In [2]:
# =========================
# 1. IMPORT LIBRARY
# =========================
import pandas as pd
import numpy as np
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression

# =========================
# 2. LOAD DATA
# =========================
df = sns.load_dataset('titanic')

# Ambil fitur penting
df = df[['survived','pclass','sex','age','sibsp','parch','fare','embarked']]

# =========================
# 3. PREPROCESSING (FIX ERROR)
# =========================

# Handle missing values (TANPA inplace)
df['age'] = df['age'].fillna(df['age'].mean())
df['embarked'] = df['embarked'].fillna(df['embarked'].mode()[0])

# Encoding categorical
le = LabelEncoder()
df['sex'] = le.fit_transform(df['sex'])
df['embarked'] = le.fit_transform(df['embarked'])

# Cek apakah masih ada NaN
print("Cek missing values:\n", df.isna().sum())

# =========================
# 4. SPLIT DATA
# =========================
X = df.drop('survived', axis=1)
y = df['survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# =========================
# 5. MODEL 1 - DECISION TREE
# =========================
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

# =========================
# 6. MODEL 2 - RANDOM FOREST (TUNING)
# =========================
rf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

# =========================
# 7. MODEL 3 - GRADIENT BOOSTING (FIX NA)
# =========================
gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
gb.fit(X_train, y_train)
y_pred_gb = gb.predict(X_test)

# =========================
# 8. MODEL 4 - STACKING (3 MODEL)
# =========================
estimators = [
    ('dt', DecisionTreeClassifier()),
    ('rf', RandomForestClassifier(n_estimators=50)),
    ('gb', GradientBoostingClassifier())
]

stack = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression()
)

stack.fit(X_train, y_train)
y_pred_stack = stack.predict(X_test)

# =========================
# 9. EVALUASI
# =========================
def evaluate(y_test, y_pred):
    return {
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1 Score': f1_score(y_test, y_pred)
    }

results = pd.DataFrame({
    'Decision Tree': evaluate(y_test, y_pred_dt),
    'Random Forest': evaluate(y_test, y_pred_rf),
    'Gradient Boosting': evaluate(y_test, y_pred_gb),
    'Stacking': evaluate(y_test, y_pred_stack)
})

print("\nHasil Perbandingan Model:\n")
print(results)

Cek missing values:
 survived    0
pclass      0
sex         0
age         0
sibsp       0
parch       0
fare        0
embarked    0
dtype: int64

Hasil Perbandingan Model:

           Decision Tree  Random Forest  Gradient Boosting  Stacking
Accuracy        0.793296       0.815642           0.810056  0.810056
Precision       0.746667       0.836066           0.812500  0.822581
Recall          0.756757       0.689189           0.702703  0.689189
F1 Score        0.751678       0.755556           0.753623  0.750000
